# Lab 6: Pandas Time Series for Streamflow and Weather

        **Week:** Week 6

        **Lab type:** Individual lab

        **Estimated time:** 2 lab periods

        ## Learning objectives

        - Parse dates.
- Filter time ranges.
- Group by month and season.
- Compare precipitation and streamflow.

        ## Earth and environmental motivation

        Time-series tools help compare precipitation, temperature, and streamflow across months and seasons.

        ## Dataset

        Weather and streamflow processed CSV files

        ## Python concepts used

        - Datetime
- Filtering
- Groupby
- Resampling
- Hydrographs

## Lab 5 Debrief and Collaborative Debugging (First 10 Minutes)

Open the debrief card from Lab 5. Two to four students or groups will
share a solved problem, an unresolved problem with evidence, or a verification
choice. Work one unresolved problem together, then report to the class.

- 0-5 min: student discussion. Compare cards in small groups and
  debug one unresolved problem together.
- 5-10 min: student reports. Two to four groups report, and the
  class records one reusable lesson.


## Required imports and project paths

Run this cell first. It finds the project root whether the notebook is opened from the
repository root or from a notebook folder.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data folder exists: {PROCESSED_DIR.exists()}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

weather = pd.read_csv(PROCESSED_DIR / "iowa_city_weather_daily.csv", parse_dates=["date"])
stream = pd.read_csv(PROCESSED_DIR / "iowa_streamflow_daily.csv", parse_dates=["date"])
weather["month"] = weather["date"].dt.month
stream["month"] = stream["date"].dt.month

In [ ]:
monthly_weather = weather.groupby("month").agg(temp_mean_c=("temp_mean_c", "mean"), precipitation_mm=("precipitation_mm", "sum"))
monthly_stream = stream.groupby("month").agg(discharge_cfs=("discharge_cfs", "mean"))
combined = monthly_weather.join(monthly_stream)
print(combined)

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.bar(combined.index, combined["precipitation_mm"], color="skyblue", label="Precipitation")
ax1.set_xlabel("Month")
ax1.set_ylabel("Precipitation (mm)")
ax2 = ax1.twinx()
ax2.plot(combined.index, combined["discharge_cfs"], color="navy", marker="o", label="Discharge")
ax2.set_ylabel("Mean discharge (cfs)")
ax1.set_title("Monthly precipitation and streamflow")
fig.tight_layout()
plt.show()

## Guided coding: resampling with a datetime index

`resample` needs the date as the index. Note that the aggregation choice is
scientific, not cosmetic: monthly temperature should be averaged, monthly
precipitation should be summed.

In [ ]:
weather_indexed = weather.set_index("date")
monthly = weather_indexed.resample("MS").agg({"temp_mean_c": "mean", "precipitation_mm": "sum"})
print(monthly.head())

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(monthly.index, monthly["precipitation_mm"], marker="o", markersize=3)
ax.set_xlabel("Month")
ax.set_ylabel("Precipitation (mm/month)")
ax.set_title("Monthly precipitation totals, Iowa City")
fig.tight_layout()
plt.show()

## Guided coding: rolling means and anomalies

A 30-day rolling mean smooths out storms. Subtracting each month's long-term
average (the climatology) turns the series into anomalies, which show whether
a period was unusually wet or dry for its season.

In [ ]:
stream_indexed = stream.set_index("date").sort_index()
stream_indexed["rolling_30d"] = stream_indexed["discharge_cfs"].rolling(30, min_periods=15).mean()

monthly_climatology = stream_indexed.groupby(stream_indexed.index.month)["discharge_cfs"].mean()
stream_indexed["climatology"] = stream_indexed.index.month.map(monthly_climatology)
stream_indexed["anomaly_cfs"] = stream_indexed["discharge_cfs"] - stream_indexed["climatology"]

one_year = stream_indexed.loc["2024"]
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.axhline(0, color="gray", linewidth=0.8)
ax.plot(one_year.index, one_year["anomaly_cfs"], color="teal")
ax.set_xlabel("Date")
ax.set_ylabel("Discharge anomaly (cfs)")
ax.set_title("2024 daily discharge minus the monthly climatology")
fig.tight_layout()
plt.show()

## Guided coding: how does the river respond to big rain?

For the five largest daily rainfalls, the loop below records the flow on the
event day, the peak flow within the following five days, and how many days
the peak lagged the rain.

In [ ]:
merged = weather[["date", "precipitation_mm"]].merge(stream[["date", "discharge_cfs"]], on="date", how="inner")
top_events = merged.nlargest(5, "precipitation_mm")
flow_by_date = merged.set_index("date")["discharge_cfs"]

rows = []
for event_date in top_events["date"]:
    window = flow_by_date.loc[event_date : event_date + pd.Timedelta(days=5)]
    rows.append({
        "event_date": event_date.date(),
        "rain_mm": float(merged.loc[merged["date"] == event_date, "precipitation_mm"].iloc[0]),
        "flow_day0_cfs": float(window.iloc[0]),
        "flow_peak_cfs": float(window.max()),
        "days_to_peak": int(window.to_numpy().argmax()),
    })
response = pd.DataFrame(rows)
print(response)

## Try it yourself

Filter the data to one year and compare the monthly pattern with the full record.

## Graded Checkpoint: Independent Analysis

The guided cells are examples. Complete the task below with your own code; an
unchanged guided notebook does not meet the submission requirement.

Create a calendar-month table containing total precipitation and mean discharge. Identify the wettest month, report its discharge, and make a two-panel figure. Verify that the monthly index is ordered and contains no duplicate months.


In [ ]:
# GRADED CHECKPOINT
# Write your code below. Include at least one verification check.


### Scientific Explanation

Replace this text with your interpretation. State what the result means, cite
one piece of numerical or graphical evidence, and name one limitation.


## More practice

1. Compute the mean temperature for each season (winter, spring, summer,
   fall) using a month-to-season mapping.
2. Find the single wettest calendar month in the record (year and month,
   for example 2019-05) using the monthly totals.
3. Compute the correlation between monthly precipitation and monthly mean
   discharge. Is it stronger than the daily relationship, and why would
   that be?

## Common mistakes and debugging tips

- Check that `PROCESSED_DIR.exists()` printed `True`.
- Read error messages from the bottom upward.
- Check column names with `df.columns` before selecting a column.
- Keep units in figure labels and written interpretations.
- Re-run earlier cells after changing data-loading or helper-code cells.

## Deliverables checklist

        - [ ] Monthly weather and streamflow summaries
- [ ] One comparison figure
- [ ] Short interpretation

        ## Short reflection

        Why might precipitation and streamflow peaks occur in different months?

        ## Rubric summary

        Correctness and completion, readable code, labeled figures, interpretation,
        and reproducibility all matter. Your submitted notebook should run from top to bottom.

## Debrief Card for the Next Lab

Complete this before the next Wednesday meeting. An unresolved problem is a
valid and useful report.

**Goal:** Replace this text with what you were trying to calculate or show.

**Expected result:** Replace this text.

**What happened:** Replace this text with the result, error, or design choice.

**Evidence:** Include an error message, value, figure observation, or tiny test.

**What I tried:** Replace this text.

**Fix or next check:** State what fixed it, or what the class should test next.

**Lesson from a classmate:** Complete this during the next debrief.
